In [1]:
import os
import requests
import json
import numpy as np
import tensorflow as tf
from tqdm import tqdm
import torch
import torch.nn as nn

#LLM Utils (for weights loading)

In [2]:
def download_and_load_gpt2(model_size, models_dir):
    # Validate model size
    allowed_sizes = ("124M", "355M", "774M", "1558M")
    if model_size not in allowed_sizes:
        raise ValueError(f"Model size not in {allowed_sizes}")

    # Define paths
    model_dir = os.path.join(models_dir, model_size)
    base_url = "https://openaipublic.blob.core.windows.net/gpt-2/models"
    filenames = [
        "checkpoint", "encoder.json", "hparams.json",
        "model.ckpt.data-00000-of-00001", "model.ckpt.index",
        "model.ckpt.meta", "vocab.bpe"
    ]

    # Download files
    os.makedirs(model_dir, exist_ok=True)
    for filename in filenames:
        file_url = os.path.join(base_url, model_size, filename)
        file_path = os.path.join(model_dir, filename)
        download_file(file_url, file_path)

    ## We have reached here until now ---> we have downloaded the files on our local machine.

    # Load settings and params
    tf_ckpt_path = tf.train.latest_checkpoint(model_dir)
    settings = json.load(open(os.path.join(model_dir, "hparams.json")))
    params = load_gpt2_params_from_tf_ckpt(tf_ckpt_path, settings)

    return settings, params

def download_file(url, destination):
    try:
        # Send a GET request to download the file, disabling SSL verification
        response = requests.get(url, stream=True, verify=False)

        # Get the total file size from headers, defaulting to 0 if not present
        file_size = int(response.headers.get("content-length", 0))

        # Check if file exists and has the same size
        if os.path.exists(destination):
            file_size_local = os.path.getsize(destination)
            if file_size == file_size_local:
                print(f"File already exists and is up-to-date: {destination}")
                return

        # Define the block size for reading the file
        block_size = 1024  # 1 Kilobyte

        # Initialize the progress bar with total file size
        progress_bar_description = url.split("/")[-1]  # Extract filename from URL
        with tqdm(total=file_size, unit="iB", unit_scale=True, desc=progress_bar_description) as progress_bar:
            # Open the destination file in binary write mode
            with open(destination, "wb") as file:
                # Iterate over the file data in chunks
                for chunk in response.iter_content(block_size):
                    progress_bar.update(len(chunk))  # Update progress bar
                    file.write(chunk)  # Write the chunk to the file

    except requests.exceptions.RequestException as e:
        print(f"Error downloading the file: {e}")
        print(f"Please check the URL: {url}")

def load_gpt2_params_from_tf_ckpt(ckpt_path, settings):
    # Initialize parameters dictionary with empty blocks for each layer
    params = {"blocks": [{} for _ in range(settings["n_layer"])]}

    # Iterate over each variable in the checkpoint
    for name, _ in tf.train.list_variables(ckpt_path):
        # Load the variable and remove singleton dimensions
        variable_array = np.squeeze(tf.train.load_variable(ckpt_path, name))

        # Process the variable name to extract relevant parts
        variable_name_parts = name.split("/")[1:]  # Skip the 'model/' prefix

        # Identify the target dictionary for the variable
        target_dict = params
        if variable_name_parts[0].startswith("h"):
            layer_number = int(variable_name_parts[0][1:])
            target_dict = params["blocks"][layer_number]

        # Recursively access or create nested dictionaries
        for key in variable_name_parts[1:-1]:
            target_dict = target_dict.setdefault(key, {})

        # Assign the variable array to the last key
        last_key = variable_name_parts[-1]
        target_dict[last_key] = variable_array

    return params

In [3]:
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

import numpy as np

def load_weights_into_gpt(gpt, params):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params['wpe'])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params['wte'])

    for b in range(len(params["blocks"])):
        q_w, k_w, v_w = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T)

        q_b, k_b, v_b = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b)

        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight,
            params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias,
            params["blocks"][b]["attn"]["c_proj"]["b"])

        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight,
            params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias,
            params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight,
            params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias,
            params["blocks"][b]["mlp"]["c_proj"]["b"])

        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale,
            params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift,
            params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale,
            params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift,
            params["blocks"][b]["ln_2"]["b"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])

In [4]:
settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
checkpoint: 100%|██████████| 77.0/77.0 [00:00<00:00, 232kiB/s]
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
encoder.json: 100%|██████████| 1.04M/1.04M [00:00<00:00, 3.01MiB/s]
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verificat

In [5]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 256, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

# Define model configurations in a dictionary for compactness
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

# Copy the base configuration and update with specific model settings
model_name = "gpt2-small (124M)"  # Example model name
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})
NEW_CONFIG

{'vocab_size': 50257,
 'context_length': 1024,
 'emb_dim': 768,
 'n_heads': 12,
 'n_layers': 12,
 'drop_rate': 0.1,
 'qkv_bias': True}

In [6]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

#Inference utils

In [7]:
#checking the simple text generation

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # add batch dimension
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # remove batch dimension
    return tokenizer.decode(flat.tolist())

"""temp of: 1 means that only one token would be prbable for selection among others with lower probabilites
            0.1 means that higher confidence; meaning that only one will have higher probability
             1.5 means lower probability meaning that multiple tokens can be sampled as they might have the same probability """
#temperature scaling (greater than 0: 1=no change, 0.1=higher confidence, 1.5=lower confidence): logits ---> logits/temperature ---> softmax ---> multinomial sampling
#top-K: logits ---> top-k (makes the elements other than top-k as -inf) ---> logits/temperature ---> softmax ---> multinomial sampling

#both are combined

def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        if top_k is not None:
            # Keep only top_k values
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1] # select the last element i.e., the smallest from each batch's output
            logits = torch.where(logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits)

        # New: Apply temperature scaling
        if temperature > 0.0:
            logits = logits / temperature

            # Apply softmax to get probabilities
            probs = torch.softmax(logits, dim=-1)  # (batch_size, context_len)

            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # (batch_size, 1)

        # Otherwise same as before: get idx of the vocab entry with the highest logits value
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch_size, 1)

        if idx_next == eos_id:  # Stop generating early if end-of-sequence token is encountered and eos_id is specified
            break

        # Same as before: append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch_size, num_tokens+1)

    return idx

#Original LLM Architecture

In [8]:
class MultiheadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        #step 3
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask",torch.triu(torch.ones(context_length, context_length), diagonal=1))


    def forward(self, x):
        b, num_tokens, d_in = x.shape

        #step 4
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        #step 5
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        #step 6
        keys = keys.transpose(1,2)
        queries = queries.transpose(1,2)
        values = values.transpose(1,2)

        #step 7
        attn_scores = queries @ keys.transpose(2,3)

        #step 8
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        #step 9 - 11
        ctx_vec = (attn_weights @ values).transpose(1, 2)

        #step 12
        ctx_vec = ctx_vec.contiguous().view(b, num_tokens, self.d_out)
        ctx_vec = self.out_proj(ctx_vec)

        return ctx_vec

#==========================================================================


class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

#==========================================================================


class GeLU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0/torch.pi)) * (x + 0.044715 * torch.pow(x,3))))

#==========================================================================


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]),
            GeLU(),
            nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"])
        )
    def forward(self, x):
        return self.layers(x)

#==========================================================================

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiheadAttention(
            d_in = cfg["emb_dim"],
            d_out = cfg["emb_dim"],
            context_length = cfg["context_length"],
            dropout = cfg["drop_rate"],
            num_heads = cfg["n_heads"],
            qkv_bias = cfg["qkv_bias"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x

#=======================================================================

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.tok_emb = nn.Embedding(config["vocab_size"], config["emb_dim"])
        self.pos_emb = nn.Embedding(config["context_length"], config["emb_dim"])
        self.drop_emb = nn.Dropout(config["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(config) for _ in range(config["n_layers"])]
        )

        self.final_norm = LayerNorm(config["emb_dim"])
        self.out_head = nn.Linear(config["emb_dim"], config["vocab_size"], bias=False)

    def forward(self, x):
        batch_size, seq_len = x.shape
        tok_embeddings = self.tok_emb(x) #[2,4,768]
        pos_embeddings = self.pos_emb(torch.arange(seq_len, device=x.device)) #[2,4,768]
        x = tok_embeddings + pos_embeddings #[2,4,768]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x) #[2,4,50257]
        return logits

In [9]:
DEVICE = torch.device('cuda:0')
model_original = GPT(NEW_CONFIG).to(DEVICE).float()
load_weights_into_gpt(model_original, params)
model_original.eval();

#Triton powered Fused LLM Architecture

##Kernel (LayerNorm)

In [10]:
import triton
import triton.language as tl
DEVICE = torch.device('cuda:0')

In [11]:
# fetching the properties of the GPU
properties = triton.runtime.driver.active.utils.get_device_properties(DEVICE.index)
NUM_SM = properties["multiprocessor_count"]
NUM_REGS = properties["max_num_regs"] # total number of registers on a single SM
TOTAL_SRAM_PER_SM = properties["max_shared_mem"]
WARP_SIZE = properties["warpSize"]

print(properties)

{'max_shared_mem': 65536, 'max_num_regs': 65536, 'multiprocessor_count': 40, 'warpSize': 32, 'sm_clock_rate': 1590000, 'mem_clock_rate': 5001000, 'mem_bus_width': 256}


In [12]:
@triton.jit
def _layernorm_forward(
    x_ptr, y_ptr, weight_ptr, bias_ptr,
    mean_ptr, rstd_ptr,
    stride_M, N, eps,
    BLOCK_SIZE:tl.constexpr
):
  # ---------- FASTER APPROACH -----------
  row = tl.program_id(0)
  x_ptr += row * stride_M
  y_ptr += row * stride_M

  cols = tl.arange(0, BLOCK_SIZE)
  mask = cols < N

  x = tl.load(x_ptr + cols, mask=mask, other=0.0).to(tl.float32)

  mean = tl.sum(x, axis=0) / N

  diff = tl.where(cols < N, x - mean, 0.0)

  var = tl.sum(diff*diff, axis=0) / N
  rstd = 1 / (tl.sqrt(var + eps))

  tl.store(mean_ptr + row, mean)
  tl.store(rstd_ptr + row, rstd)

  w = tl.load(weight_ptr + cols, mask = mask, other=0.0)
  b = tl.load(bias_ptr + cols, mask = mask, other=0.0)

  x_normed = diff * rstd
  y = x_normed * w + b
  tl.store(y_ptr + cols, y, mask=mask)

In [13]:
@triton.jit
def _layernorm_backward_dldx(
    x_ptr, dldx_ptr, dldy_ptr, w_ptr,
    mean_ptr, rstd_ptr,
    stride, N,
    locks_ptr, dldw_intermediate_ptr, dldb_intermediate_ptr,
    BLOCK_SIZE_N: tl.constexpr,
    GROUP_SIZE: tl.constexpr
):
  PID = tl.program_id(0)
  cols = tl.arange(0, BLOCK_SIZE_N)
  mask = cols < N
  x_ptr += PID * stride #calculate the starting position of the particular row handled by the PID by jumping "stride" elements
  dldx_ptr += PID * stride
  dldy_ptr += PID * stride

  #loading data into SRAM
  x = tl.load(x_ptr + cols, mask=mask, other=0.0).to(tl.float32)
  dldy = tl.load(dldy_ptr + cols, mask=mask, other=0.0).to(tl.float32)
  w = tl.load(w_ptr + cols, mask=mask).to(tl.float32)
  mean = tl.load(mean_ptr + PID)
  rstd = tl.load(rstd_ptr + PID)

  #computation of dldx
  x_normalized = tl.where(mask, (x - mean)*rstd, 0.0) # shape(BLOCKSIZE_N)
  dydx_norm = tl.where(mask, dldy * w, 0.0) # shape(BLOCKSIZE_N)
  c1 = tl.sum(x_normalized * dydx_norm, axis=0)/N # shape(1)
  c2 = tl.sum(dydx_norm, axis=0)/N # shape(1)
  dldx = (dydx_norm - (c1 * x_normalized + c2)) * rstd #shape(BLOCKSIZE_N)

  #writing back to DRAM
  tl.store(dldx_ptr + cols, dldx, mask=mask)

  #----------------dl/dx calculation done------------------------------

  dldw_contribution = (dldy * x_normalized).to(w.dtype)
  dldb_contribution = (dldy).to(w.dtype)

  lock_id = PID % GROUP_SIZE
  locks_ptr += lock_id # gives the address of locks[lock_id]
  # address of the second half of the locks array corresponding to the lock_id e.g., lock_id = 0 then count_ptr = 0 + 8 = 8
  count_ptr = locks_ptr + GROUP_SIZE

  dldw_inter_ptrs = dldw_intermediate_ptr + lock_id * N + cols # base address + pointers to the particular row to write data to
  dldb_inter_ptrs = dldb_intermediate_ptr + lock_id * N + cols

  # (lock[lock_id], expected, new)
  # if at lock[lock_id], the value read by a PID is 0, then it writes 1 at lock[lock_id]
  # returns the previous value (i.e., 0) and then checks in the while loop if 0 == 1 which becomes false after which loop is exited
  while tl.atomic_cas(locks_ptr, 0, 1) == 1:
    #if another PID tries to access it while it is locked, then the value read at lock[lock_id] will be 1 which wont be the expected 0
    # so it writes nothing at that location of the locks array and returns 1 which makes the condition true and that PID will be stuck in the loop
    # waiting for the previous PID to unlock it using tl.xchg(locks_ptr, 0) at the end of the kernel after it finishes writing
    pass

  count = tl.load(count_ptr) # fetch the value of the number of counts data has been written
  if count == 0: #no other PID has written to this row
    #write the value as 1 so that the other ptr knows that it has previously been written
    #so it first reads it, add and then write it again to that row
    tl.atomic_xchg(count_ptr, 1)

  else: #if its previosuly been written, add the previous values
    dldw_contribution += tl.load(dldw_inter_ptrs, mask=mask)
    dldb_contribution += tl.load(dldb_inter_ptrs, mask=mask)

  tl.store(dldw_inter_ptrs, dldw_contribution, mask=mask)
  tl.store(dldb_inter_ptrs, dldb_contribution, mask=mask)

  tl.atomic_xchg(locks_ptr, 0) # unlock after writing

In [14]:
@triton.jit
def _layernorm_backward_dldw_dldb(
    dldw_intermediate_ptr, dldb_intermediate_ptr,
    dldw_ptr, dldb_ptr,
    GROUP_SIZE, N,
    BLOCK_SIZE_M: tl.constexpr,
    BLOCK_SIZE_N: tl.constexpr
):
  PID = tl.program_id(0) # splitted wrt columns
  # since BLOCK_SIZE_N = 32 and there are PIDs from 0 to 3
  # PID 0 --> cols = [0,.....,31]
  # PID 1 --> cols = [32,.....,63]
  # PID 2 --> cols = [64,.....,95]
  # PID 3 --> cols = [96,.....,128]
  col_ptrs = PID * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N) # each PID handling respective columns

  #accumulating the values as tiles
  dldw_acc = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N),dtype=tl.float32)
  dldb_acc = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N),dtype=tl.float32)

  #suppose BLOCK_SIZE_M = 32 & GROUP_SIZE = 64
  # loop iterates through [0, 32, 64 (not included)] => 2 iterations; i = 0 & i = 32
  # i = 0 (1st iter) -> row_ptrs = [0,.....,BLOCK_SIZE_M] = [0,.....,31]
  # i = 32 (2nd iter) -> row_ptrs = [0+i,......,BLOCK_SIZE_M + i] = [32,.....,63]
  for i in range(0, GROUP_SIZE, BLOCK_SIZE_M):
    row_ptrs = i + tl.arange(0, BLOCK_SIZE_M) # i + [0,......,31]

    #row_ptrs[:,None] = shape(BLOCK_SIZE_M, 1)
    #col_ptrs[None,:] = shape(1, BLOCK_SIZE_N)
    #2D mask is shape (BLOCK_SIZE_M, BLOCK_SIZE_N) is created
    #checking against the actual dimensions which is GROUP_SIZE and N
    #because BLOCK_SIZE_M and BLOCK_SIZE_N would always return true for the mask
    #mask only comes in handy if either row_ptrs & col_ptrs have values exceeding the GROUP_SIZE or N respectively
    mask = (row_ptrs[:,None] < GROUP_SIZE) & (col_ptrs[None,:] < N)

    # row_ptrs -> shape(BLOCK_SIZE_M, 1) = (32,1)
    # col_ptrs -> shape(1, BLOCK_SIZE_N) = (1,32)

    # row_ptrs[:, None] * N:                  # shape (32, 1) * 512

    # [[0   * 512],     [[0   ],
    #  [1   * 512],  =   [512 ],
    #  [2   * 512],      [1024],
    #  ...                ...
    #  [31  * 512]]      [15872]]

    # col_ptrs[None, :]:                      # shape (1, 32)
    # [[0, 1, 2, 3, ..., 31]]

    # Adding them together via broadcasting:
    # [[0    + 0,   0    + 1,   0    + 2,   ..., 0    + 31],   ← row 0
    #  [512  + 0,   512  + 1,   512  + 2,   ..., 512  + 31],   ← row 1
    #  [1024 + 0,   1024 + 1,   1024 + 2,   ..., 1024 + 31],   ← row 2
    #  ...
    #  [15872+ 0,   15872+ 1,   15872+ 2,   ..., 15872+ 31]]   ← row 31

    # Final offsets (32, 32):
    # [[0,    1,    2,    ..., 31  ],
    #  [512,  513,  514,  ..., 543  ],
    #  [1024, 1025, 1026, ...,  1055],
    #  ...
    #  [15872,15873,15874,..., 15893]]
    offsets = row_ptrs[:,None] * N + col_ptrs[None,:]

    dldw_acc += tl.load(dldw_intermediate_ptr + offsets, mask=mask, other=0.)
    dldb_acc += tl.load(dldb_intermediate_ptr + offsets, mask=mask, other=0.)


  sum_dLdw = tl.sum(dldw_acc, axis=0) # shape (BLOCK_SIZE_N)
  sum_dLdb = tl.sum(dldb_acc, axis=0)

  # Write the final sum computed by each PID to the output.
  tl.store(dldw_ptr + col_ptrs, sum_dLdw, mask=col_ptrs < N)
  tl.store(dldb_ptr + col_ptrs, sum_dLdb, mask=col_ptrs < N)

In [15]:
class LayerNormFunction(torch.autograd.Function):
  @staticmethod
  def forward(ctx, x, normalized_shape, weight, bias, eps):
    # x = [32, 512, 768] --> (32 Sentences, 512 Words each, 768 Hidden Units)
    # x.shape[-1] = 768 (last dimension)
    # x.reshape(-1, 768) => x = [32 x 512, 768] = [16384, 768]
    # M = 16384, N = 768
    M, N = x.reshape(-1, x.shape[-1]).shape

    # store the output and the mean and (reciprocal) std deviation (rstd) on the DRAM instead of SRAM for backward pass computation
    y = torch.empty_like(x)
    mean = torch.empty((M,), dtype=torch.float32, device=x.device)
    rstd = torch.empty((M,), dtype=torch.float32, device=x.device)

    # maximum number of elements (float32 or some other type) of each row that can be held in SRAM of each SM
    # given each SM's SRAM is of capacity "TOTAL_SRAM_PER_SM"
    MAX_FUSED_SIZE = TOTAL_SRAM_PER_SM // x.element_size()

    # make the blocksize to handle the elements not exceeding the max each SM can handle
    BLOCK_SIZE = min(MAX_FUSED_SIZE, triton.next_power_of_2(N))

    if N > BLOCK_SIZE:
      raise RuntimeError("this layernorm does not support features dim >= 64kb")

    # keep the warps for each block to be between 1 and 8 with a scaling factor of 8
    num_warps = min(max(BLOCK_SIZE // 256, 1), 8)

    # the grid can have programs equal to the number of rows
    # _layernorm_forward[(M,)](
    #     x, y,
    #     weight, bias,
    #     mean, rstd,
    #     x.stride(0), N, eps,
    #     BLOCK_SIZE=BLOCK_SIZE, num_warps=num_warps
    # )

    row_stride = x.stride(-2)
    _layernorm_forward[(M,)](
        x, y,
        weight, bias,
        mean, rstd,
        row_stride, N, eps, # Use row_stride here
        BLOCK_SIZE=BLOCK_SIZE, num_warps=num_warps
    )

    ctx.save_for_backward(x, weight, bias, mean, rstd)
    ctx.BLOCK_SIZE = BLOCK_SIZE
    ctx.num_warps = num_warps
    ctx.eps = eps

    return y


  @staticmethod
  def backward(ctx, dLdy):
    x,w,b,mean,rstd = ctx.saved_tensors
    M, N = x.reshape(-1, x.shape[-1]).shape

    dLdx = torch.empty_like(x) #(M,N)
    dLdw = torch.empty_like(w) #(N)
    dLdb = torch.empty_like(b) #(N)

    GROUP_SIZE = 64
    if N <= 8192: GROUP_SIZE = 96
    if N <= 4096: GROUP_SIZE = 128
    if N <= 1024: GROUP_SIZE = 256

    dldw_intermediate = torch.zeros((GROUP_SIZE, N), dtype=x.dtype, device=w.device)
    dldb_intermediate = torch.zeros((GROUP_SIZE, N), dtype=x.dtype, device=w.device)

    locks = torch.zeros(2*GROUP_SIZE, dtype=torch.int32, device=w.device)

    # _layernorm_backward_dldx[(M,)](
    #     x, dLdx, dLdy,
    #     w, mean, rstd,
    #     x.stride(0), N,
    #     locks, dldw_intermediate, dldb_intermediate,
    #     BLOCK_SIZE_N = ctx.BLOCK_SIZE,
    #     GROUP_SIZE = GROUP_SIZE,
    #     num_warps = ctx.num_warps,
    # )

    row_stride = x.stride(-2)
    _layernorm_backward_dldx[(M,)](
        x, dLdx, dLdy,
        w, mean, rstd,
        row_stride, N, # Use row_stride here
        locks, dldw_intermediate, dldb_intermediate,
        BLOCK_SIZE_N = ctx.BLOCK_SIZE,
        GROUP_SIZE = GROUP_SIZE,
        num_warps = ctx.num_warps,
    )

    # second kernel to be parallelized across the cols
    # for example if N = 128 and BLOCK_SIZE_N = 32
    # grid would have 4 PIDs
    # pid 0 -> handles COLS [0:32]
    # pid 1 -> handles COLS [32:64]
    # pid 2 -> handles COLS [64:96]
    # pid 3 -> handles COLS [96:128]

    grid = lambda meta : [triton.cdiv(N, meta["BLOCK_SIZE_N"])]
    _layernorm_backward_dldw_dldb[grid](
        dldw_intermediate, dldb_intermediate,
        dLdw, dLdb,
        min(GROUP_SIZE, M), N,
        BLOCK_SIZE_M=32, BLOCK_SIZE_N = 128, # to handle each tile of intermediate matrices
    )

    return dLdx, None, dLdw, dLdb, None

In [16]:
layernorm = LayerNormFunction.apply
def test_layernorm_kernel(M, N, dtype, eps=1e-5, device=DEVICE):
  x = -2.3 + 0.5 * torch.randn((M,N), dtype=dtype, device=device) #normal distribution scaled by 0.5 and shifted 2.3 to the right
  x.requires_grad_(True)
  weight = torch.rand((N,), dtype=dtype, device=device, requires_grad=True)
  bias = torch.rand((N,), dtype=dtype, device=device, requires_grad=True)

  # forward pass
  y_tri = layernorm(x, (N,), weight, bias, eps)
  y_ref = torch.nn.functional.layer_norm(x, (N,), weight, bias, eps).to(dtype)
  torch.testing.assert_close(y_tri, y_ref, atol=1e-2, rtol=0)
  print("PASSED FORWARD PASS")

  # backward pass
  dLdy = 0.1 * torch.randn_like(x) # sample gradient of the loss wrt layernorm output

  y_tri.backward(dLdy, retain_graph=True) # calculates dl/dx, dl/dw, dl/db
  dldx_tri, dldw_tri, dldb_tri = [_.grad.clone() for _ in [x, weight, bias]]
  x.grad, weight.grad, bias.grad = None, None, None # clear the computed gradients

  y_ref.backward(dLdy, retain_graph=True) # calculates dl/dx, dl/dw, dl/db
  dldx_ref, dldw_ref, dldb_ref = [_.grad.clone() for _ in [x, weight, bias]]


  torch.testing.assert_close(dldx_tri, dldx_ref, atol=1e-2, rtol=0)
  torch.testing.assert_close(dldw_tri, dldw_ref, atol=1e-2, rtol=0)
  torch.testing.assert_close(dldb_tri, dldb_ref, atol=1e-2, rtol=0)
  print("PASSED BACKWARD PASS")

In [17]:
test_layernorm_kernel(1151, 8192, torch.float16)

PASSED FORWARD PASS
PASSED BACKWARD PASS


##LLM definition

In [18]:
class MultiheadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        #step 3
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask",torch.triu(torch.ones(context_length, context_length), diagonal=1))


    def forward(self, x):
        b, num_tokens, d_in = x.shape

        #step 4
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        #step 5
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        #step 6
        keys = keys.transpose(1,2)
        queries = queries.transpose(1,2)
        values = values.transpose(1,2)

        #step 7
        attn_scores = queries @ keys.transpose(2,3)

        #step 8
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        #step 9 - 11
        ctx_vec = (attn_weights @ values).transpose(1, 2)

        #step 12
        ctx_vec = ctx_vec.contiguous().view(b, num_tokens, self.d_out)
        ctx_vec = self.out_proj(ctx_vec)

        return ctx_vec

#==========================================================================


# class LayerNorm(nn.Module):
#     def __init__(self, emb_dim):
#         super().__init__()
#         self.eps = 1e-5
#         self.scale = nn.Parameter(torch.ones(emb_dim))
#         self.shift = nn.Parameter(torch.zeros(emb_dim))

#     def forward(self, x):
#         mean = x.mean(dim=-1, keepdim=True)
#         var = x.var(dim=-1, keepdim=True, unbiased=False)
#         norm_x = (x - mean) / torch.sqrt(var + self.eps)
#         return self.scale * norm_x + self.shift

class LayerNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-5):
        super().__init__()
        self.eps = eps

        self.scale = nn.Parameter(torch.ones(emb_dim)) #weights
        self.shift = nn.Parameter(torch.zeros(emb_dim)) #bias
        self.normalized_shape = (emb_dim,)

    def forward(self, x):
        if not x.is_contiguous():
            x = x.contiguous()

        # Passing self.weight and self.bias to your Triton autograd function
        return layernorm(x, self.normalized_shape, self.scale, self.shift, self.eps)

#==========================================================================


class GeLU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0/torch.pi)) * (x + 0.044715 * torch.pow(x,3))))

#==========================================================================


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]),
            GeLU(),
            nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"])
        )
    def forward(self, x):
        return self.layers(x)

#==========================================================================

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiheadAttention(
            d_in = cfg["emb_dim"],
            d_out = cfg["emb_dim"],
            context_length = cfg["context_length"],
            dropout = cfg["drop_rate"],
            num_heads = cfg["n_heads"],
            qkv_bias = cfg["qkv_bias"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x

#=======================================================================

class GPT_triton(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.tok_emb = nn.Embedding(config["vocab_size"], config["emb_dim"])
        self.pos_emb = nn.Embedding(config["context_length"], config["emb_dim"])
        self.drop_emb = nn.Dropout(config["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(config) for _ in range(config["n_layers"])]
        )

        self.final_norm = LayerNorm(config["emb_dim"])
        self.out_head = nn.Linear(config["emb_dim"], config["vocab_size"], bias=False)

    def forward(self, x):
        batch_size, seq_len = x.shape
        tok_embeddings = self.tok_emb(x) #[2,4,768]
        pos_embeddings = self.pos_emb(torch.arange(seq_len, device=x.device)) #[2,4,768]
        x = tok_embeddings + pos_embeddings #[2,4,768]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x) #[2,4,50257]
        return logits

In [19]:
model_triton = GPT_triton(NEW_CONFIG).to(DEVICE).float()
load_weights_into_gpt(model_triton, params)
model_triton.to(DEVICE);
model_triton.eval();

#LLM Inferences

In [20]:
idx = text_to_token_ids("Every effort moves you", tokenizer).to(DEVICE)

In [21]:
torch.manual_seed(123)

token_ids = generate(
    model=model_original.to(DEVICE),
    # idx=text_to_token_ids("Every effort moves you", tokenizer),
    idx=idx,
    max_new_tokens=25,
    context_size=NEW_CONFIG["context_length"],
    top_k=50,
    temperature=0.5
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you as far as you can.

When you're ready, you have to go.

You have to go.


In [22]:
torch.manual_seed(123)

token_ids = generate(
    model=model_triton,
    # idx=text_to_token_ids("Every effort moves you", tokenizer).to(DEVICE),
    idx=idx,
    max_new_tokens=25,
    context_size=NEW_CONFIG["context_length"],
    top_k=50,
    temperature=0.5
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you as far as you can.

When you're ready, you have to go.

You have to go.


In [28]:
import triton

# Measure the time for a single forward pass
ms_torch = triton.testing.do_bench(lambda: model_original(idx))
ms_triton = triton.testing.do_bench(lambda: model_triton(idx))

speedup = ms_torch / ms_triton
print(f"PyTorch: {ms_torch:.3f} ms")
print(f"Triton:  {ms_triton:.3f} ms")
print(f"Speedup: {speedup:.2f}x")

PyTorch: 23.768 ms
Triton:  16.215 ms
Speedup: 1.47x


In [49]:
import time

def benchmark_llm(model, idx, max_new_tokens):
    # Warmup
    for _ in range(0,11):
      _ = model(idx)
    torch.cuda.synchronize()

    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    # Measure TTFT
    start_event.record()
    logits = model(idx)
    # (Simplified: logic to pick 1 token would go here)
    end_event.record()
    torch.cuda.synchronize()
    ttft = start_event.elapsed_time(end_event)

    # Measure total time for max_new_tokens
    # (Wrap your generate function call here)
    t0 = time.perf_counter()
    # _ = generate(model, idx, max_new_tokens=max_new_tokens)

    _ = generate(
                    model=model_triton,
                    # idx=text_to_token_ids("Every effort moves you", tokenizer).to(DEVICE),
                    idx=idx,
                    max_new_tokens=max_new_tokens,
                    context_size=NEW_CONFIG["context_length"],
                    top_k=50,
                    temperature=0.5
                )

    torch.cuda.synchronize()
    total_time = (time.perf_counter() - t0) * 1000 # to ms

    itl = (total_time - ttft) / (max_new_tokens - 1)
    tps = max_new_tokens / (total_time / 1000)

    return ttft, itl, tps

# # Run for both
# stats_pt = benchmark_llm(model_original, idx, 25)
# stats_tri = benchmark_llm(model_triton, idx, 25)

# for label, stats in [("PyTorch", stats_pt), ("Triton", stats_tri)]:
#     ttft, itl, tps = stats
#     print(f"{label} → TTFT: {ttft:.2f}ms | ITL: {itl:.2f}ms/tok | TPS: {tps:.2f} tok/s")

In [56]:
import torch

# --- HEADER & CONFIG ---
print("="*65)
print("Standard Pytorch LLM vs Triton LLM (LayerNorm kernel)")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print("="*65)

# --- RUN BENCHMARKS ---
stats_pt = benchmark_llm(model_original, idx, 25)
stats_tri = benchmark_llm(model_triton, idx, 25)

# --- RESULTS TABLE ---
for label, stats in [("PyTorch LLM", stats_pt), ("Triton LLM", stats_tri)]:
    ttft, itl, tps = stats
    # print(f"{label:<12} => TTFT: {ttft:>6.2f}ms | ITL: {itl:>6.2f}ms/tok | TPS: {tps:>6.2f} tok/s")
    print(f"--- {label} ---")
    print(f"Time to First Token (TTFT):   {ttft:>6.2f} ms")
    print(f"Inter-Token Latency (ITL):    {itl:>6.2f} ms/token")
    print(f"Token Per Second (TPS):       {tps:>6.2f} tokens/sec")
    print()

# --- SUMMARY ---
speedup = stats_tri[2] / stats_pt[2]
print("-" * 65)
print(f"{speedup:.2f}x Throughput Increase (Tokens/Sec)")
print("="*65)

Standard Pytorch LLM vs Triton LLM (LayerNorm kernel)
GPU: Tesla T4
--- PyTorch LLM ---
Time to First Token (TTFT):    20.99 ms
Inter-Token Latency (ITL):     22.96 ms/token
Token Per Second (TPS):        43.71 tokens/sec

--- Triton LLM ---
Time to First Token (TTFT):    16.43 ms
Inter-Token Latency (ITL):     15.04 ms/token
Token Per Second (TPS):        66.23 tokens/sec

-----------------------------------------------------------------
1.52x Throughput Increase (Tokens/Sec)
